# TMDB Master Pipeline Orchestration

Runs the full medallion pipeline end-to-end — Raw-to-Bronze, Bronze-to-Silver, Silver-to-Gold — sequentially via `dbutils.notebook.run()`. Accepts two optional parameters: `load_type` (default `incremental`) and `batch_id` (default blank = all batches). Each stage logs its own run to `workspace.tmdb.pipeline_execution_logs`.

In [0]:
from datetime import datetime

# ---- Master pipeline parameters (optional widgets) ----
dbutils.widgets.text("load_type", "incremental", "Load type: 'full' or 'incremental'")
dbutils.widgets.text("batch_id", "", "Batch ID (blank = process all batches)")

load_type = dbutils.widgets.get("load_type").strip().lower()
batch_id  = dbutils.widgets.get("batch_id").strip()

if load_type not in ("full", "incremental"):
    raise ValueError(f"Invalid load_type '{load_type}'. Use 'full' or 'incremental'.")

print("Master pipeline parameters")
print(f"  load_type : {load_type}")
print(f"  batch_id  : {batch_id if batch_id else '(blank — all batches)'}")

# ---- Pipeline stages (executed in order) ----
# Relative paths resolve from this notebook's location (project root).
PIPELINE_STAGES = [
    ("Raw-to-Bronze",    "bronze/bronze_injestion"),
    ("Bronze-to-Silver", "silver/silver_injestion"),
    ("Silver-to-Gold",   "gold/gold_transformation"),
]

# Max seconds allowed per stage before dbutils.notebook.run times out
STAGE_TIMEOUT_SECONDS = 3600

In [0]:
pipeline_start = datetime.now()
stage_results = []
current_stage = None

try:
    for stage_name, stage_path in PIPELINE_STAGES:
        current_stage = stage_name
        print("=" * 70)
        print(f"[{stage_name}] START — {stage_path}")
        print(f"  load_type={load_type} | batch_id={batch_id if batch_id else '(all)'}")
        print("=" * 70)

        stage_start = datetime.now()

        dbutils.notebook.run(
            stage_path,
            STAGE_TIMEOUT_SECONDS,
            {"load_type": load_type, "batch_id": batch_id},
        )

        stage_seconds = round((datetime.now() - stage_start).total_seconds(), 2)
        stage_results.append({"stage": stage_name, "path": stage_path,
                              "status": "SUCCESS", "seconds": stage_seconds})
        print(f"[{stage_name}] SUCCESS in {stage_seconds}s\n")

    total_seconds = round((datetime.now() - pipeline_start).total_seconds(), 2)
    print("=" * 70)
    print(f"MASTER PIPELINE COMPLETED — {total_seconds}s total")
    print("=" * 70)
    for r in stage_results:
        print(f"  {r['stage']:<18} {r['status']:<8} {r['seconds']:>10}s  ({r['path']})")

except Exception as e:
    elapsed = round((datetime.now() - pipeline_start).total_seconds(), 2)
    print("=" * 70)
    print(f"MASTER PIPELINE FAILED at stage '{current_stage}' after {elapsed}s")
    print(f"Error: {e}")
    print("=" * 70)
    for r in stage_results:
        print(f"  {r['stage']:<18} {r['status']:<8} {r['seconds']:>10}s  ({r['path']})")
    raise